# 00 — Environment and Data Access Checks

This notebook validates runtime environment, repository paths, and required source assets before any fitting.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
paths=[l.strip() for l in SMEFT_LISTING.read_text().splitlines() if l.strip()]
print('listed paths:', len(paths))
print('dictionary exists:', LOCAL_DICT.exists())

In [ ]:
REQUIRED_MODULES=['numpy','scipy','matplotlib','pandas','awkward','uproot','hist','coffea']
missing=[]
for m in REQUIRED_MODULES:
    try:
        __import__(m); print('ok',m)
    except Exception as e:
        missing.append((m,str(e)))
        print('missing',m,e)
if missing:
    print('Install:', 'pip install ' + ' '.join([m for m,_ in missing]))

In [ ]:
REQUIRED_PATHS=[
'/uscms_data/d3/azhou/smeft/analysis/stxs_functions.py',
'/uscms_data/d3/azhou/smeft/analysis/vbfprocessor.py',
'/uscms_data/d3/azhou/smeft/analysis/hbb-coffea/boostedhiggs/__init__.py',
'/uscms_data/d3/azhou/smeft/analysis/coffea',
]
for rp in REQUIRED_PATHS:
    hit = any(p==rp or p.startswith(rp+'/') for p in paths)
    print(hit, rp)

In [ ]:
# Save a machine-readable environment report for downstream notebooks
report={
 'repo': str(REPO),
 'listing': str(SMEFT_LISTING),
 'local_dictionary': str(LOCAL_DICT),
 'has_local_dictionary': LOCAL_DICT.exists(),
}
(NOTEBOOK_DIR/'env_report.json').write_text(json.dumps(report, indent=2))
print('wrote', NOTEBOOK_DIR/'env_report.json')